# Day 16 — NumPy fundamentals: arrays, vectorization, broadcasting
Objectives:
- Create ndarrays and understand dtypes/shapes.
- Vectorize operations to replace Python loops.
- Use broadcasting effectively.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-16`. Read
`python/ds-60day/companion-guides/day16_numpy_fundamentals.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A NumPy array is a rectangular, usually homogeneous block of values.
Its `shape` states the length of each dimension, its `ndim` counts
dimensions, and its `dtype` controls representation and supported
numeric behavior. Unlike a nested Python list, an array supports
elementwise operations and axis-aware reductions.

Vectorization expresses work as array operations implemented in
optimized compiled loops. Broadcasting compares shapes from the right
and virtually expands dimensions of length one or missing leading
dimensions. It avoids manual loops, but a broadcastable expression can
still be semantically wrong; always state what each axis represents.

### Vocabulary

- **array:** a multidimensional homogeneous data container.
- **shape:** the length of every array dimension.
- **dtype:** the stored scalar representation, such as `int64` or `float64`.
- **axis:** a dimension along which an operation is applied.
- **vectorization:** expressing elementwise work as array operations.
- **broadcasting:** NumPy's rule for combining compatible unequal shapes.

## Syntax anatomy

For a shape `(rows, columns)` array, `values.mean(axis=0)` collapses the
row axis and returns one mean per column; `axis=1` collapses columns and
returns one mean per row. In `matrix - column_means`, shapes `(r, c)`
and `(c,)` align from the right, so the one-dimensional vector is used
across every row.

### Worked example 1 — Inspect shape before calculating

Give each dimension a meaning and compare axis reductions. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import numpy as np

scores = np.array([[8, 6, 10], [4, 9, 8]])
{
    "shape": scores.shape,
    "per_column": scores.mean(axis=0).tolist(),
    "per_row": scores.mean(axis=1).tolist(),
}

**Expected observation:** `{'shape': (2, 3), 'per_column': [6.0, 7.5, 9.0], 'per_row': [8.0, 7.0]}`.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Center every column through broadcasting

A length-three vector applies to the three columns of every row. Predict first; then run the next cell.

In [ ]:
column_means = scores.mean(axis=0)
centered = scores - column_means
(centered.tolist(), centered.mean(axis=0).round(10).tolist())

**Expected observation:** The centered rows are displayed and every column mean is `[0.0, 0.0, 0.0]` up to floating-point precision.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Print `.shape`, `.dtype`, and a small slice before diagnosing an array calculation.
2. Name axis meanings in prose before choosing `axis=0` or `axis=1`.
3. When broadcasting fails, align shapes from the right and insert an explicit singleton dimension if needed.
4. Distinguish a view from a copy before mutating a slice.

**Alternative to compare:** Use Python lists for small heterogeneous general-purpose data; use arrays for rectangular numeric computation.

**Boundary to test:** Empty axes, integer overflow, NaN propagation, boolean-mask shape mismatch, and accidental view mutation need deliberate handling.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import numpy as np
a = np.array([1,2,3], dtype=np.int64)
b = np.array([0.1,0.2,0.3], dtype=np.float64)
a.dtype, b.dtype, a.shape, b.shape

# Vectorization vs loops
x = np.arange(1_000_000)
%timeit x * 2

# Broadcasting
M = np.ones((3,3))
v = np.arange(3)
M + v


## Common operations
- Slicing, views vs copies.
- Aggregations and axis param.


In [ ]:
Z = np.arange(12).reshape(3,4)
Z, Z[:, 1:3], Z.sum(axis=0), Z.mean(axis=1)


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Create a 3×4 array containing integers 0 through 11, then compute row sums, column means, and the maximum of each row.
   **Expected behavior:** shapes are `(3, 4)`, `(3,)`, `(4,)`, and `(3,)` for the original and three results. **Constraint:** use vectorized reductions with explicit axes, not Python loops.
   **Verify:** assert both shapes and exact values.

2. Generate two deterministic arrays with `np.random.default_rng(42)`, perform elementwise addition/multiplication, matrix multiplication where shapes permit, and boolean-mask selection. **Constraints:** write the shape equation before `@` and do not use the legacy global random state.
   **Verify:** rerunning from a fresh kernel produces identical arrays and results.

3. Demonstrate broadcasting by standardizing each column of a small 2-D array.
   **Expected behavior:** each standardized column has mean approximately 0 and standard deviation approximately 1. **Constraints:** keep means/stds as `(columns,)`, reject zero-standard-deviation columns, and assert the final shape equals the input shape.
   **Verify:** Assert output shape equals input shape, nonconstant columns have mean near zero and standard deviation near one, and zero-variance input raises or follows the written policy.

### Additional mastery practice

Reason about shape, axis, dtype, and view/copy semantics before applying vectorized operations. Verify results with small arrays.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

4. **Prediction:** Predict the shape and values of a `(2, 3)` array plus a `(3,)` array, then explain broadcasting alignment.
   **Progressive hint:** Broadcasting compares dimensions from the right.
   **Verify:** Assert the `(2, 3) + (3,)` result shape and exact values; write which right-aligned dimension proves the broadcast is valid.
5. **Tracing:** Trace `array.sum(axis=0)` and `array.sum(axis=1)` by naming which dimension is removed and what each output element represents.
   **Progressive hint:** The named axis is the dimension reduced.
   **Verify:** Calculate both reductions by hand and assert `axis=0` removes rows into one value per column while `axis=1` removes columns into one per row.
6. **Implementation:** Implement column standardization that leaves zero-variance columns as zeros instead of dividing by zero.
   **Progressive hint:** Replace a zero standard deviation with a safe denominator.
   **Verify:** Assert nonconstant columns standardize near mean 0/std 1, zero-variance columns become exactly zeros, and shape/dtype policy is preserved.
7. **Debugging:** Repair code that mutates an original array through a slice when an independent working array was intended.
   **Progressive hint:** Use `.copy()` at the ownership boundary.
   **Verify:** Record `np.shares_memory` before repair, mutate the working slice, and assert `.copy()` keeps the original array unchanged.
8. **Edge case and explanation:** Demonstrate integer overflow with a small integer dtype and prevent it by selecting a wider dtype before arithmetic.
   **Progressive hint:** The array dtype, not Python's unbounded integer behavior, controls storage.
   **Verify:** Show the small-dtype wrapped result, repeat after casting to a sufficiently wide dtype, and assert the widened arithmetic matches Python's expected integer.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Create a 3×4 array containing integers 0 through 11, then compute row sums, column means, and the maximum of each row. **Expected behavior:** shapes are `(3, 4)`, `(3,)`, `(4,)`, and `(3,)` for the original and three results. **Constraint:** use vectorized reductions with explicit axes, not Python loops. **Verify:** assert both shapes and exact values.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Create a 3×4 array containing integers 0 through 11, then compute row sums, column means, and the maximum of each row. shapes are `(3, 4)`, `(3,)`, `(4,)`, and `(3,)` for the or...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Generate two deterministic arrays with `np.random.default_rng(42)`, perform elementwise addition/multiplication, matrix multiplication where shapes permit, and boolean-mask selection. **Constraints:** write the shape equation before `@` and do not use the legacy global random state. **Verify:** rerunning from a fresh kernel produces identical arrays and results.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Generate two deterministic arrays with `np.random.default_rng(42)`, perform elementwise addition/multiplication, matrix multiplication where shapes permit, and boolean-mask sele...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** Demonstrate broadcasting by standardizing each column of a small 2-D array. **Expected behavior:** each standardized column has mean approximately 0 and standard deviation approximately 1. **Constraints:** keep means/stds as `(columns,)`, reject zero-standard-deviation columns, and assert the final shape equals the input shape. **Verify:** Assert output shape equals input shape, nonconstant columns have mean near zero and standard deviation near one, and zero-variance input raises or follows the written policy.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Demonstrate broadcasting by standardizing each column of a small 2-D array. each standardized column has mean approximately 0 and standard deviation approximately 1. keep means/...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict the shape and values of a `(2, 3)` array plus a `(3,)` array, then explain broadcasting alignment. **Progressive hint:** Broadcasting compares dimensions from the right. **Verify:** Assert the `(2, 3) + (3,)` result shape and exact values; write which right-aligned dimension proves the broadcast is valid.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Predict the shape and values of a `(2, 3)` array plus a `(3,)` array, then explain broadcasting alignment. Broadcasting compares dimensions from the right. Assert the `(2, 3) +...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace `array.sum(axis=0)` and `array.sum(axis=1)` by naming which dimension is removed and what each output element represents. **Progressive hint:** The named axis is the dimension reduced. **Verify:** Calculate both reductions by hand and assert `axis=0` removes rows into one value per column while `axis=1` removes columns into one per row.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Trace `array.sum(axis=0)` and `array.sum(axis=1)` by naming which dimension is removed and what each output element represents. The named axis is the dimension reduced. Calculat...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement column standardization that leaves zero-variance columns as zeros instead of dividing by zero. **Progressive hint:** Replace a zero standard deviation with a safe denominator. **Verify:** Assert nonconstant columns standardize near mean 0/std 1, zero-variance columns become exactly zeros, and shape/dtype policy is preserved.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Implement column standardization that leaves zero-variance columns as zeros instead of dividing by zero. Replace a zero standard deviation with a safe denominator. Assert noncon...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair code that mutates an original array through a slice when an independent working array was intended. **Progressive hint:** Use `.copy()` at the ownership boundary. **Verify:** Record `np.shares_memory` before repair, mutate the working slice, and assert `.copy()` keeps the original array unchanged.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Repair code that mutates an original array through a slice when an independent working array was intended. Use `.copy()` at the ownership boundary. Record `np.shares_memory` bef...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 8 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Demonstrate integer overflow with a small integer dtype and prevent it by selecting a wider dtype before arithmetic. **Progressive hint:** The array dtype, not Python's unbounded integer behavior, controls storage. **Verify:** Show the small-dtype wrapped result, repeat after casting to a sufficiently wide dtype, and assert the widened arithmetic matches Python's expected integer.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 8 — your work
# Short contract: Demonstrate integer overflow with a small integer dtype and prevent it by selecting a wider dtype before arithmetic. The array dtype, not Python's unbounded integer behavior, co...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
